# SENTINEL · Notebook 1 · `routing/`

**What this notebook builds:** the **Routing Model** from slide 4:
> *Dijkstra / A\* on the OpenStreetMap road graph; blocked edges removed, flooded edges time-penalised, matrix rebuilt on each closure.*

### How to use this notebook
1. Run the grey code boxes **one at a time, from top to bottom**. To run one, click the **▶ button** at its left edge.
2. Wait until it finishes and shows a **✅ message** before you run the next one.
3. **Red text with the word `Error`** = stop and send a screenshot to Claude.
   Orange or pink *warnings* are normal. Ignore them.
4. If Colab says *"This notebook was not authored by Google"*, click **Run anyway**.

## Step 1 · Install the tools
- **OSMnx** downloads real roads from OpenStreetMap.
- **NetworkX** contains the Dijkstra and A\* route-finding algorithms.
- **Folium** draws interactive maps.

In [1]:
!pip install -q "osmnx>=2.0,<3" folium
import math, json, os, random
import networkx as nx
import osmnx as ox
import pandas as pd
import folium
print("OSMnx version:", ox.__version__)
print("✅ Step 1 done: tools installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.2 MB/s eta 0:00:00
OSMnx version: 2.1.1
✅ Step 1 done: tools installed


## Step 2 · Connect your Google Drive
Colab forgets files when you close it, so we save everything in your Google Drive, in a folder called **SENTINEL**.

A pop-up will ask for permission: choose your Google account → **Continue** → **Allow** (tick all boxes if asked).

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/SENTINEL/data"
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Step 2 done: files will be saved in Google Drive → SENTINEL → data")

Mounted at /content/drive
✅ Step 2 done: files will be saved in Google Drive → SENTINEL → data


## Step 3 · Choose the disaster area
We use the **Kurla–Sion belt** along the Mithi River in Mumbai, a low-lying area that was among the worst hit in the July 2005 floods.
The area is a circle of 3 km around the point below.

In [3]:
CENTER = (19.0700, 72.8800)   # (latitude, longitude): Kurla–Sion, Mumbai
RADIUS_M = 3000               # metres around the centre
print("✅ Step 3 done: area centre", CENTER, "| radius", RADIUS_M, "m")

✅ Step 3 done: area centre (19.07, 72.88) | radius 3000 m


## Step 4 · Download the real road network
- Every **junction** becomes a *node*; every **road segment** becomes an *edge*.
- Each edge gets a **travel_time** in seconds = length ÷ speed.
- Where OpenStreetMap has no speed limit, we use the speeds below. These are **our assumption** for ambulances in Mumbai monsoon traffic; you can change them.
- We keep only the part of the network where every junction can reach every other one, so routes always exist before the disaster.

In [4]:
print("Downloading roads from OpenStreetMap. This takes 1–3 minutes...")
G = ox.graph_from_point(CENTER, dist=RADIUS_M, network_type="drive")

CITY_SPEEDS_KPH = {   # assumed ambulance speeds in Mumbai monsoon traffic
    "motorway": 50, "trunk": 40, "primary": 30, "secondary": 25,
    "tertiary": 20, "residential": 15, "unclassified": 15, "living_street": 10,
}
G = ox.routing.add_edge_speeds(G, hwy_speeds=CITY_SPEEDS_KPH, fallback=15)
G = ox.routing.add_edge_travel_times(G)
G = ox.truncate.largest_component(G, strongly=True)
print(f"✅ Step 4 done: {G.number_of_nodes():,} junctions and {G.number_of_edges():,} road segments")

✅ Step 4 done: 3,609 junctions and 7,931 road segments


## Step 5 · Find the real hospitals
OpenStreetMap marks hospitals with the tag `amenity=hospital`.
Bed counts are not public in OpenStreetMap, so capacities will be **assumed** in Notebook 2 and labelled as synthetic.

In [5]:
gdf = ox.features_from_point(CENTER, tags={"amenity": "hospital"}, dist=RADIUS_M)
hospitals, seen = [], set()
for _, row in gdf.iterrows():
    name = row.get("name")
    if not isinstance(name, str) or name in seen:
        continue
    seen.add(name)
    point = row.geometry.representative_point()
    hospitals.append({"id": f"H{len(hospitals) + 1}", "name": name,
                      "lat": float(point.y), "lon": float(point.x)})
hospitals = hospitals[:12]          # keep at most 12 hospitals for the demo
for h in hospitals:                 # connect each hospital to its nearest road junction
    h["node"] = int(ox.distance.nearest_nodes(G, X=h["lon"], Y=h["lat"]))

print(pd.DataFrame(hospitals)[["id", "name", "lat", "lon"]].to_string(index=False))
if len(hospitals) < 4:
    print("⚠️ Few hospitals found. In Step 3 change RADIUS_M to 4000, then run Steps 3, 4 and 5 again.")
else:
    print(f"✅ Step 5 done: {len(hospitals)} real hospitals found")

 id                                 name       lat       lon
 H1                         JOY Hospital 19.055877 72.897395
 H2                      Inlaks Hospital 19.046613 72.896422
 H3            Ameeta Patil Nursing Home 19.062225 72.899729
 H4     Mata Ramabai B Ambedkar Hospital 19.055714 72.894641
 H5                         maa hospital 19.058506 72.890405
 H6                   Kurla Nursing Room 19.070817 72.880679
 H7                    new life hospital 19.089761 72.905000
 H8                    muktabai hospital 19.094602 72.903291
 H9 v care hospital intensive& unit care 19.063478 72.881788
H10                   arpan nursing home 19.069393 72.878120
H11                     fauziya hospital 19.062758 72.873176
H12                      arayan hospital 19.063610 72.881927
✅ Step 5 done: 12 real hospitals found


## Step 6 · The Routing Model: Dijkstra and A\*
- **Dijkstra** spreads out from the start junction, always extending the quickest route found so far, until it reaches the hospital. It always finds the fastest route.
- **A\*** does the same, but it also aims toward the hospital using a guess: *straight-line distance ÷ top speed*. That guess is never more than the real time, so A\* gives the **same answer** while checking fewer junctions.

We test it on a casualty near Kurla station.

In [6]:
def straight_line_m(lat1, lon1, lat2, lon2):
    # Distance in metres between two points on Earth (haversine formula)
    r = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

TOP_SPEED_MS = max(d["speed_kph"] for _, _, d in G.edges(data=True)) / 3.6

def astar_guess(u, v):
    # A* heuristic: straight-line distance ÷ top speed (never more than the real time)
    a, b = G.nodes[u], G.nodes[v]
    return straight_line_m(a["y"], a["x"], b["y"], b["x"]) / TOP_SPEED_MS

def fastest_route(graph, start, end, method="astar"):
    # Returns (list of junctions, minutes). If no road connects them: (None, infinity)
    try:
        if method == "dijkstra":
            path = nx.shortest_path(graph, start, end, weight="travel_time", method="dijkstra")
        else:
            path = nx.astar_path(graph, start, end, heuristic=astar_guess, weight="travel_time")
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None, float("inf")
    seconds = sum(min(d["travel_time"] for d in graph.get_edge_data(u, v).values())
                  for u, v in zip(path[:-1], path[1:]))
    return path, seconds / 60

PATIENT = (19.0656, 72.8792)   # a casualty near Kurla station
p_node = int(ox.distance.nearest_nodes(G, X=PATIENT[1], Y=PATIENT[0]))

target = None
for h in hospitals:            # pick a hospital that is a real drive away (not next door)
    path_d, min_d = fastest_route(G, p_node, h["node"], "dijkstra")
    if path_d and len(path_d) >= 6:
        target = h
        break
path_a, min_a = fastest_route(G, p_node, target["node"], "astar")

print(f"Casualty near Kurla station → {target['name']}")
print(f"  Dijkstra: {min_d:.1f} min")
print(f"  A*      : {min_a:.1f} min  (same answer, fewer junctions checked)")
print("✅ Step 6 done: the Routing Model works")

Casualty near Kurla station → JOY Hospital
  Dijkstra: 9.0 min
  A*      : 9.0 min  (same answer, fewer junctions checked)
✅ Step 6 done: the Routing Model works


## Step 7 · Disaster strikes: flood + road closure
Two rules from slide 4:
- **Flooded edges are time-penalised:** every road inside the flood zone becomes **3× slower**. The zone below is *illustrative*, placed near the Mithi River.
- **Blocked edges are removed:** we close a road segment in the middle of the old route (think: a bridge that is no longer safe), which forces a re-route.

The original map `G` stays untouched; all damage is applied to a copy called `G_live`.

In [7]:
FLOOD_CENTER = (19.0705, 72.8705)  # illustrative flood zone near the Mithi River
FLOOD_RADIUS_M = 700
FLOOD_FACTOR = 3.0                 # flooded roads are 3x slower

def apply_flood(graph, center, radius_m, factor):
    flooded = 0
    for u, v, k, d in graph.edges(keys=True, data=True):
        mid_lat = (graph.nodes[u]["y"] + graph.nodes[v]["y"]) / 2
        mid_lon = (graph.nodes[u]["x"] + graph.nodes[v]["x"]) / 2
        if straight_line_m(mid_lat, mid_lon, center[0], center[1]) <= radius_m:
            d["travel_time"] *= factor
            d["flooded"] = True
            flooded += 1
    return flooded

def close_road(graph, u, v):
    # Remove a road segment in both directions (e.g. an unsafe bridge)
    for a, b in [(u, v), (v, u)]:
        while graph.has_edge(a, b):
            graph.remove_edge(a, b)

G_live = G.copy()
n_flooded = apply_flood(G_live, FLOOD_CENTER, FLOOD_RADIUS_M, FLOOD_FACTOR)
mid = len(path_a) // 2
CLOSED = (path_a[mid - 1], path_a[mid])   # a road segment in the middle of the old route
close_road(G_live, *CLOSED)

new_path, new_min = fastest_route(G_live, p_node, target["node"])
print(f"Flooded road segments: {n_flooded}")
if new_path:
    print(f"Before: {min_a:.1f} min  →  after flood + closure: {new_min:.1f} min")
    print("✅ Step 7 done: SENTINEL re-routed around the damage")
else:
    print("No road left to this hospital. SENTINEL would send the patient elsewhere (Notebook 4).")

Flooded road segments: 394
Before: 9.0 min  →  after flood + closure: 9.2 min
✅ Step 7 done: SENTINEL re-routed around the damage


## Step 8 · See it on a map
- **Blue** = route before the disaster · **Orange** = new route · **Black** = closed road
- **Light-blue circle** = flood zone · **Red markers** = real hospitals · **Orange marker** = casualty

Zoom with your mouse wheel; hover over lines and markers for details.

In [10]:
def coords(graph, path):
    return [(graph.nodes[n]["y"], graph.nodes[n]["x"]) for n in path]

m = folium.Map(location=CENTER, zoom_start=14, tiles="cartodbpositron")
folium.Circle(FLOOD_CENTER, radius=FLOOD_RADIUS_M, color="#38BDF8", fill=True,
              fill_opacity=0.25, tooltip="Flood zone (illustrative)").add_to(m)
for h in hospitals:
    folium.Marker((h["lat"], h["lon"]), tooltip=f"{h['id']} · {h['name']}",
                  icon=folium.Icon(color="red", icon="plus")).add_to(m)
folium.Marker(PATIENT, tooltip="Casualty", icon=folium.Icon(color="orange", icon="user")).add_to(m)
folium.PolyLine(coords(G, path_a), color="#1E6FD9", weight=6, opacity=0.8,
                tooltip=f"Before: {min_a:.1f} min").add_to(m)
if new_path:
    folium.PolyLine(coords(G, new_path), color="#F59E0B", weight=6, opacity=0.9,
                    tooltip=f"After: {new_min:.1f} min").add_to(m)
folium.PolyLine(coords(G, list(CLOSED)), color="black", weight=10, tooltip="Closed road").add_to(m)
m

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## Step 9 · The travel-time matrix
Slide 4 says the matrix is *"rebuilt on each closure"*. This table is what the optimiser (Notebook 4) reads.
- **Rows:** 5 ambulance bases (placed at random junctions for now; Notebook 2 places them properly).
- **Columns:** hospitals. **Values:** minutes to drive there.

Compare the two tables: times go up near the flood, and some routes change completely.

In [9]:
random.seed(7)
bases = random.sample(list(G.nodes), 5)

def travel_matrix(graph, sources, hospitals):
    rows = {}
    for i, s in enumerate(sources):
        times = nx.single_source_dijkstra_path_length(graph, s, weight="travel_time")
        rows[f"Base {i + 1}"] = {h["id"]: (round(times[h["node"]] / 60, 1) if h["node"] in times else None)
                                 for h in hospitals}
    return pd.DataFrame(rows).T

before = travel_matrix(G, bases, hospitals)
after = travel_matrix(G_live, bases, hospitals)
print("Minutes BEFORE the flood:")
print(before.to_string())
print("\nMinutes AFTER flood + closure (matrix rebuilt):")
print(after.to_string())
print("\n✅ Step 9 done: travel-time matrix rebuilt after the closure")

Minutes BEFORE the flood:
         H1    H2   H3   H4   H5    H6    H7    H8   H9   H10   H11  H12
Base 1  8.0   9.9  8.8  7.2  7.8  13.8  18.3  19.6  8.6  14.4  16.7  8.6
Base 2  8.2  11.8  7.8  8.1  9.1   4.0   9.1  10.3  6.7   2.8   3.7  6.7
Base 3  4.8   8.4  4.4  4.8  5.8   6.1  10.8  12.1  6.6   6.6   9.0  6.6
Base 4  2.4   5.0  4.1  2.4  4.6  11.2  15.2  16.5  6.0  11.8  14.1  6.0
Base 5  2.2   2.5  3.9  1.3  3.3   9.9  14.1  15.4  4.7  10.4  12.8  4.7

Minutes AFTER flood + closure (matrix rebuilt):
         H1    H2   H3   H4   H5    H6    H7    H8   H9   H10   H11  H12
Base 1  8.0   9.9  8.8  7.2  7.8  13.8  18.3  19.6  8.6  14.4  17.7  8.6
Base 2  9.1  12.7  8.7  9.0  9.9   5.5   9.9  11.1  7.5   5.0   8.3  7.5
Base 3  5.0   8.6  4.6  4.9  5.8   8.3  10.8  12.1  7.7   8.8  12.1  7.7
Base 4  2.4   5.0  4.1  2.4  4.6  11.2  15.2  16.5  6.0  11.8  15.1  6.0
Base 5  2.2   2.5  3.9  1.3  3.3   9.9  14.1  15.4  4.7  10.4  13.7  4.7

✅ Step 9 done: travel-time matrix rebuilt after 

## Step 10 · Save everything to Google Drive
Three files go into **Google Drive → SENTINEL → data**. Notebook 2 loads them automatically.

In [11]:
ox.io.save_graphml(G, f"{DATA_DIR}/mumbai_roads.graphml")
with open(f"{DATA_DIR}/hospitals.json", "w") as f:
    json.dump(hospitals, f, indent=2)
with open(f"{DATA_DIR}/area.json", "w") as f:
    json.dump({"center": CENTER, "radius_m": RADIUS_M,
               "flood_center": FLOOD_CENTER, "flood_radius_m": FLOOD_RADIUS_M,
               "flood_factor": FLOOD_FACTOR, "speeds_kph": CITY_SPEEDS_KPH}, f, indent=2)
size_mb = os.path.getsize(f"{DATA_DIR}/mumbai_roads.graphml") / 1e6
print(f"✅ Step 10 done: 3 files saved (road file is {size_mb:.1f} MB)")
print("🎉 Notebook 1 complete. Send Claude a screenshot of the map (Step 8) and the tables (Step 9).")

✅ Step 10 done: 3 files saved (road file is 3.8 MB)
🎉 Notebook 1 complete. Send Claude a screenshot of the map (Step 8) and the tables (Step 9).


## What you just built (say this to a judge)
> *"Our routing engine loads the real OpenStreetMap road network for Kurla–Sion. Every road has a travel time. When a road is closed we delete it, and when an area floods we multiply its travel times, then we re-run A\*, which gives the same answer as Dijkstra but searches less. The travel-time matrix between ambulances and hospitals is rebuilt after every closure, and that matrix feeds our optimiser."*

| Slide 4 says | Where it is in this notebook |
|---|---|
| OpenStreetMap road graph · OSMnx | Step 4 |
| Dijkstra / A\* | Step 6 |
| Flooded edges time-penalised | Step 7 (`apply_flood`) |
| Blocked edges removed | Step 7 (`close_road`) |
| Travel-time matrix rebuilt on each closure | Step 9 |